## Summary

This notebook processes and concatenates mapped RPK (Reads Per Kilobase) data across all source plates for Lassa fever PhIP-Seq experiments. It loads, standardizes by sample name (with box location), and merges the data from multiple plates, deduplicates samples by adding unique identifiers, integrates metadata (so hearing loss data ) which will be converted to an anndata metadata files. 

Concatenating all rpk files

FC over AG over average AG value across all plates!

AND

renaming sample names

In [21]:
import pandas as pd
import re

f1 = pd.read_csv('downloads_study749_mapped_rpk.csv', index_col=0)
f2 = pd.read_csv('downloads_study765_mapped_rpk.csv', index_col=0)
f3 = pd.read_csv('downloads_study766_mapped_rpk.csv', index_col=0)
f4 = pd.read_csv('downloads_study773_mapped_rpk.csv', index_col=0)
f5 = pd.read_csv('downloads_study774_mapped_rpk.csv', index_col=0)
f6 = pd.read_csv('downloads_study805_mapped_rpk.csv', index_col = 0)

### drop samples with < 100,000 readcounts

In [25]:
samples_to_drop = pd.read_csv('../readcounts_studyxxx.csv/sample_list_less_than_100k_readcounts.csv').set_index('sample_uid_name')

drop_samples = samples_to_drop.index.astype(str).str.strip().tolist()

# Normalize IDs so names with/without a trailing _001 match.
def normalize_sample_id(x):
    return re.sub(r'_001$', '', str(x).strip())

def standardize_df_column_names(df):
    df = df.copy()
    df.columns = [normalize_sample_id(c) for c in df.columns]
    return df

# Standardize raw source-plate dataframe column names first.
f1 = standardize_df_column_names(f1)
f2 = standardize_df_column_names(f2)
f3 = standardize_df_column_names(f3)
f4 = standardize_df_column_names(f4)
f5 = standardize_df_column_names(f5)
f6 = standardize_df_column_names(f6)

drop_samples_norm = {normalize_sample_id(x) for x in drop_samples if str(x).strip()} #convert c to text with str(x)
#remove leading/trailing whitespace with .strip(), use that result ar a boolean test in if : non-empty string -> True
#empty string -> False, skip it

def drop_low_readcount_columns(df):
    keep_cols = [col for col in df.columns if col not in drop_samples_norm]
    return df.loc[:, keep_cols]

# Drop low-readcount samples from all raw source-plate dataframes.
f1 = drop_low_readcount_columns(f1)
f2 = drop_low_readcount_columns(f2)
f3 = drop_low_readcount_columns(f3)
f4 = drop_low_readcount_columns(f4)
f5 = drop_low_readcount_columns(f5)
f6 = drop_low_readcount_columns(f6)

In [26]:
print(f1.shape, f2.shape, f3.shape, f4.shape, f5.shape, f6.shape)

(90132, 175) (90132, 164) (90132, 170) (90132, 185) (90132, 535) (90132, 179)


below you can find all sample names as I have them standardized for plate aliquoting and plate layout for phipseq

In [27]:
target_prefixes_raw = r'''
C-135-1_1_A1
C-191-2_1_A2
C-191-3_1_A3
C-498-3_1_A4
C-497-3_1_A5
C-498-2_1_A6
C-140-1_1_A7
C-497-1_1_A8
C-497-2_1_A9
C-498-1_1_B1
C-499-3_1_B2
C-499-2_1_B3
C-456-2_1_B4
C-458-1_1_B5
C-458-3_1_B6
C-464-1_1_B7
C-457-3_1_B8
C-456-3_1_B9
C-453-2_1_C1
C-462-1_1_C2
C-475-3_1_C3
C-440-1_1_C4
C-192-1_1_C5
C-486-1_1_C6
C-486-3_1_C7
C-447-3_1_C8
C-135-3_1_C9
C-113-2_1_D1
C-135-2_1_D2
C-192-3_1_D3
C-113-1_1_D4
C-191-1_1_D5
C-115-3_1_D6
C-192-2_1_D7
C-482-1_1_D8
C-483-1_1_D9
C-178-1_1_E1
C-487-1_1_E2
C-484-1_1_E3
C-145-3_1_E4
C-138-2_1_E5
C-483-3_1_E6
C-126-1_1_E7
C-138-3_1_E8
C-484-3_1_E9
C-138-1_1_F1
C-463-3_1_F2
C-181-1_1_F3
C-183-1_1_F4
C-147-2_1_F5
C-120-2_1_F6
C-146-2_1_F7
C-448-2_1_F8
C-450-1_1_F9
C-148-1_1_G1
C-122-1_1_G2
C-148-3_1_G3
C-190-1_1_G4
C-187-2_1_G5
C-113-3_1_G6
C-125-1_1_G7
C-136-2_1_G8
C-457-1_1_G9
C-133-2_1_H1
C-150-1_1_H2
C-150-2_1_H3
C-136-1_1_H4
C-114-2_1_H5
C-452-3_1_H6
C-133-3_1_H7
C-150-3_1_H8
C-146-1_1_H9
C-130-1_1_I1
C-143-3_1_I2
C-488-3_1_I3
C-448-1_1_I4
C-439-1_1_I5
C-144-3_1_I6
C-489-3_1_I7
C-115-1_1_I8
C-149-2_1_I9
C-437-3_2_A1
C-143-2_2_A2
C-183-3_2_A3
C-112-2_2_A4
C-110-1_2_A5
C-145-2_2_A6
C-128-3_2_A7
C-137-3_2_A9
C-149-1_2_B1
C-188-3_2_B2
C-468-1_2_B3
C-188-1_2_B4
C-144-2_2_B5
C-489-1_2_B6
C-434-2_2_B7
C-151-1_2_B8
C-439-2_2_B9
C-120-3_2_C1
C-499-1_2_C2
C-499-2_2_C3
C-468-2_2_C4
C-446-1_2_C5
C-469-1_2_C6
C-445-3_2_C7
C-470-2_2_C8
C-451-3_2_C9
C-445-1_2_D1
C-461-2_2_D2
C-440-2_2_D3
C-462-2_2_D4
C-467-3_2_D5
C-467-1_2_D6
C-134-1_2_D7
C-437-2_2_D8
C-444-1_2_D9
C-442-2_2_E1
C-442-1_2_E2
C-151-2_2_E3
C-444-2_2_E4
C-128-2_2_E5
C-110-3_2_E6
C-446-2_2_E7
C-137-2_2_E8
C-472-3_2_E9
C-444-3_2_F1
C-456-1_2_F2
C-487-2_2_F3
C-463-1_2_F4
C-475-2_2_F5
C-451-2_2_F6
C-486-2_2_F7
C-470-3_2_F8
C-128-1_2_F9
C-472-2_2_G1
C-446-3_2_G2
C-143-1_2_G3
C-441-3_2_G4
C-440-3_2_G5
C-184-1_2_G6
C-461-3_2_G7
C-458-2_2_G8
C-443-2_2_G9
C-443-3_2_H1
C-437-1_2_H2
C-485-3_2_H3
C-149-3_2_H4
C-485-1_2_H5
C-485-2_2_H6
C-441-1_2_H7
C-111-1_2_H8
C-116-1_2_H9
C-116-2_2_I1
C-116-3_2_I2
C-126-2_2_I3
C-145-1_2_I4
C-190-2_2_I5
C-462-3_2_I6
C-144-1_2_I7
C-181-2_2_I8
C-181-3_2_I9
C-439-3_3_A1
C-126-3_3_A2
C-438-3_3_A3
C-438-1_3_A4
C-187-3_3_A5
C-470-1_3_A6
C-183-2_3_A8
C-134-2_3_A9
C-134-3_3_B1
C-448-3_3_B2
C-112-1_3_B3
C-112-3_3_B4
C-114-3_3_B6
C-115-2_3_B7
C-122-2_3_B8
C-122-3_3_B9
C-124-1_3_C1
C-124-3_3_C2
C-130-2_3_C3
C-136-3_3_C4
C-146-3_3_C5
C-147-1_3_C6
C-151-3_3_C7
C-438-2_3_C8
C-452-1_3_C9
C-473-2_3_D1
C-482-2_3_D2
C-482-3_3_D3
C-484-2_3_D4
C-489-2_3_D5
C-483-2_3_D6
C-190-3_3_D7
C-124-2_3_D8
C-148-2_3_D9
C-187-1_3_E1
C-447-2_3_E2
C-450-1_3_E3
C-125-2_3_E4
C-125-3_3_E5
C-441-2_3_E6
C-499-1_3_E7
C-148-1_3_E8
C-450-2_3_E9
C-461-1_3_F1
C-450-3_3_F2
C-449-1_3_F3
C-454-2_3_F4
C-488-1_3_F5
C-449-2_3_F6
C-488-2_3_F7
C-110-2_3_F8
C-464-3_3_F9
C-443-1_3_G2
C-184-3_3_G3
C-473-3_3_G4
C-454-1_3_G5
C-475-1_3_G6
C-509-2_3_G7
C-130-1_3_G8
C-509-3_3_G9
C-116-3_3_H1
C-514-1_3_H2
C-507-2_3_H3
C-514-2_3_H4
C-513-3_3_H5
C-135-1_3_H6
C-146-2_3_H7
C-510-1_3_H8
C-111-1_3_H9
C-505-3_3_I1
C-513-2_3_I2
C-512-3_3_I3
C-510-3_3_I4
C-111-2_3_I5
C-111-3_3_I6
C-138-3_3_I7
C-511-1_3_I8
C-150-1_3_I9
C-138-1_4_A1
C-511-2_4_A2
C-126-1_4_A3
C-508-3_4_A4
C-510-2_4_A5
C-506-3_4_A6
C-509-1_4_A7
C-508-2_4_A8
C-505-1_4_A9
C-146-1_4_B1
C-438-1_4_B2
C-512-2_4_B3
C-508-1_4_B4
C-192-1_4_B5
C-482-3_4_B6
C-135-3_4_B7
C-507-3_4_B8
C-187-1_4_B9
C-507-1_4_C1
C-505-2_4_C2
C-138-2_4_C3
C-184-1_4_C4
C-135-2_4_C5
C-506-2_4_C6
C-438-2_4_C7
C-146-3_4_C8
C-113-2_4_C9
C-113-1_4_D1
C-130-2_4_D2
C-513-1_4_D3
C-192-3_4_D4
C-137-2_4_D5
C-148-3_4_D6
C-130-3_4_D7
C-506-1_4_D8
C-188-3_4_D9
C-511-3_4_E1
C-115-2_4_E2
C-452-1_4_E3
C-452-3_4_E4
C-452-2_4_E5
C-451-3_4_E6
C-109-3_4_E7
C-128-2_4_E8
C-448-1_4_E9
C-150-2_4_F1
C-184-3_4_F2
C-114-3_4_F3
C-113-3_4_F4
C-191-1_4_F5
C-185-1_4_F6
C-497-3_4_F7
C-120-2_4_F8
C-191-2_4_G1
C-137-1_4_G2
C-489-1_4_G3
C-144-2_4_G4
C-497-2_4_G5
C-148-2_4_G6
C-120-1_4_G7
C-489-3_4_G8
C-439-1_4_G9
C-114-2_4_H1
C-183-1_4_H2
C-183-2_4_H3
C-144-3_4_H4
C-178-1_4_H5
C-450-1_4_H6
C-450-2_4_H7
C-450-3_4_H8
C-451-1_4_H9
C-451-2_4_I1
C-117-2_4_I2
C-117-3_4_I3
C-497-3_4_I4
C-137-3_4_I5
C-151-1_4_I6
C-506-1_4_I7
C-482-3_4_I8
C-113-3_4_I9
C-183-2_5_A1
C-112-2_5_A2
C-145-2_5_A3
C-192-2_5_A4
C-113-2_5_A5
C-148-2_5_A6
C-488-3_5_A7
C-146-3_5_A8
C-496-3_5_A9
C-111-2_5_B1
C-126-2_5_B2
C-109-1_5_B3
C-120-3_5_B4
C-109-2_5_B5
C-128-1_5_B6
C-496-3_5_B7
C-187-2_5_B8
C-178-1_5_B9
C-504-2_5_C1
C-188-2_5_C2
C-115-1_5_C3
C-136-3_5_C4
C-134-2_5_C5
C-188-1_5_C6
C-488-3_5_C8
C-512-1_5_C9
C-183-3_5_D1
C-126-3_5_D2
C-489-2_5_D3
C-482-2_5_D4
C-116-2_5_D5
C-136-1_5_D6
C-504-3_5_D7
C-504-1_5_D8
C-452-1_5_D9
C-454-1_5_E1
C-434-3_5_E2
C-440-2_5_E3
C-443-2_5_E4
C-134-2_5_E5
C-192-3_5_E6
C-130-1_5_E7
C-145-1_5_E8
C-110-3_5_E9
C-187-1_5_F1
C-509-2_5_F2
C-117-2_5_F3
C-115-3_5_F4
C-191-1_5_F5
C-124-1_5_F6
C-120-3_5_F7
C-137-1_5_F8
C-509-2_5_F9
C-192-2_5_G1
C-488-1_5_G2
C-485-1_5_G3
C-513-1_5_G4
C-488-2_5_G5
C-513-3_5_G6
C-449-1_5_G7
C-465-3_5_G8
C-446-2_5_G9
C-442-3_5_H1
C-458-1_5_H2
C-452-2_5_H3
C-465-1_5_H4
C-188-2_5_H5
C-445-1_5_H6
C-110-3_5_H7
C-188-1_5_H8
C-507-3_5_H9
C-446-1_5_I1
C-134-3_5_I2
C-445-3_5_I3
C-509-1_5_I4
C-507-2_5_I5
C-124-2_5_I6
C-463-3_5_I7
C-484-1_5_I8
C-452-3_5_I9
C-507-1_6_A1
C-110-2_6_A2
C-192-1_6_A3
C-110-1_6_A4
C-458-3_6_A5
C-496-3_6_A6
C-446-3_6_A7
C-443-2_6_A8
C-452-1_6_A9
C-443-3_6_B1
C-188-3_6_B2
C-134-1_6_B3
C-179-2_6_B4
C-510-1_6_B5
C-438-3_6_B6
C-482-1_6_B7
C-482-2_6_B8
C-482-3_6_B9
C-489-1_6_C1
C-489-2_6_C2
C-489-3_6_C3
C-496-1_6_C4
C-496-2_6_C5
C-497-1_6_C6
C-497-2_6_C7
C-497-3_6_C8
C-505-2_6_C9
C-505-3_6_D1
C-453-1_6_D2
C-438-1_6_D3
C-510-3_6_D4
C-139-2_6_D5
C-501-1_6_D6
C-500-2_6_D7
C-500-1_6_D8
C-502-2_6_D9
C-488-3_6_E1
C-506-1_6_E2
C-506-2_6_E3
C-511-1_6_E4
C-511-2_6_E5
C-513-2_6_E6
C-512-3_6_E7
C-514-2_6_E8
C-134-2_6_E9
C-111-1_6_F1
C-111-2_6_F2
C-111-3_6_F3
C-113-3_6_F4
C-124-1_6_F5
C-124-3_6_F6
C-130-1_6_F7
C-130-2_6_F8
C-137-3_6_F9
C-130-3_6_G1
C-137-1_6_G2
C-137-2_6_G3
C-147-1_6_G4
C-147-2_6_G5
C-178-1_6_G6
C-181-3_6_G7
C-184-1_6_G8
C-511-3_6_G9
C-192-3_6_H1
C-509-3_6_H2
C-506-3_6_H3
C-514-1_6_H4
C-148-3_6_H5
C-148-2_6_H6
C-148-1_6_H7
C-146-3_6_H8
C-146-1_6_H9
C-145-3_6_I1
C-145-2_6_I2
C-145-1_6_I3
C-138-1_6_I4
C-136-2_6_I5
C-136-1_6_I6
C-128-2_6_I7
C-128-1_6_I8
C-126-3_6_I9
C-126-1_7_A1
C-120-3_7_A2
C-120-1_7_A3
C-120-2_7_A4
C-116-1_7_A5
C-115-3_7_A6
C-115-2_7_A7
C-115-1_7_A8
C-114-3_7_A9
C-114-2_7_B1
C-114-1_7_B2
C-113-1_7_B3
C-113-2_7_B4
C-112-2_7_B5
C-112-1_7_B6
C-438-2_7_B7
C-150-1_7_B8
C-150-2_7_B9
C-150-3_7_C1
C-151-1_7_C2
C-151-3_7_C3
C-181-1_7_C4
C-183-2_7_C5
C-183-3_7_C6
C-187-1_7_C7
C-187-2_7_C8
C-191-1_7_C9
C-191-2_7_D1
C-191-3_7_D2
C-136-3_7_D3
C-116-2_7_D4
C-112-3_7_D5
C-183-1_7_D6
C-434-1_7_D7
C-470-2_7_D8
C-470-3_7_D9
C-444-3_7_E1
C-452-2_7_E2
C-442-3_7_E3
C-449-3_7_E4
C-442-2_7_E5
C-446-2_7_E6
C-450-3_7_E7
C-465-3_7_E8
C-124-3_7_E9
C-440-1_7_F1
C-434-2_7_F2
C-443-3_7_F3
C-446-3_7_F4
C-458-2_7_F5
C-440-3_7_F6
C-444-1_7_F7
C-444-2_7_F8
C-445-1_7_F9
C-112-1_7_G1
C-112-3_7_G2
C-470-1_7_G3
C-445-3_7_G4
C-465-1_7_G5
C-454-2_7_G6
C-112-2_7_G7
C-449-1_7_G8
C-453-1_7_G9
C-468-3_7_H1
C-466-3_7_H2
C-453-3_7_H3
C-467-2_7_H4
C-469-2_7_H5
C-469-3_7_H6
C-454-3_7_H7
C-445-2_7_H8
C-451-1_7_H9
S-111_SA1_A1
S-116_SA1_A2
S-104_SA1_A3
S-105_SA1_A4
S-115_SA1_A5
S-117_SA1_A6
S-118_SA1_A7
S-106_SA1_A8
S-107_SA1_A9
S-109_SA1_B1
S-110_SA1_B2
S-112_SA1_B3
S-113_SA1_B4
S-114_SA1_B5
S-119_SA1_B6
S-120_SA1_B7
S-120_SA1_B8
S-121_SA1_B9
S-122_SA1_C1
S-124_SA1_C2
S-125_SA1_C3
S-126_SA1_C4
S-128_SA1_C5
S-129_SA1_C6
S-130_SA1_C7
S-131_SA1_C8
S-132_SA1_C9
S-133_SA1_D1
S-134_SA1_D2
S-138_SA1_D3
S-137_SA1_D4
S-136_SA1_D5
S-135_SA1_D6
S-141_SA1_D7
S-144_SA1_D8
S-143_SA1_D9
S-142_SA1_E1
S-139_SA1_E2
S-140_SA1_E3
S-146_SA1_E4
S-145_SA1_E5
S-147_SA1_E6
S-148_SA1_E7
S-149_SA1_E8
S-150_SA1_E9
S-151_SA1_F1
S-152_SA1_F2
S-191_SA1_F3
S-192_SA1_F4
S-190_SA1_F5
S-189_SA1_F6
S-188_SA1_F7
S-187_SA1_F8
S-185_SA1_F9
S-184_SA1_G1
S-183_SA1_G2
S-186_SA1_G3
S-181_SA1_G4
S-180_SA1_G5
S-179_SA1_G6
S-178_SA1_G7
S-472_SA1_G8
S-487_SA1_G9
S-486_SA1_H1
S-488_SA1_H2
S-146_SA1_H3
S-147_SA1_H4
S-497_SA1_H5
S-498_SA1_H6
S-191_SA1_H7
S-192_SA1_H8
S-188_SA1_H9
S-473_SA1_I1
S-466_SA1_I2
S-465_SA1_I3
S-183_SA1_I4
S-130_SA1_I5
S-133_SA1_I6
S-145_SA1_I7
S-129_SA1_I8
S-143_SA1_I9
S-149_SA2_A1
S-128_SA2_A2
S-150_SA2_A3
S-137_SA2_A4
S-125_SA2_A5
S-136_SA2_A6
S-116_SA2_A7
S-111_SA2_A8
S-484_SA2_A9
S-128_SA2_B1
S-114_SA2_B2
S-168_SA2_B3
S-145_SA2_B4
S-114_SA2_B5
S-147_SA2_B6
S-115_SA2_B7
S-191_SA2_B8
S-444_SA2_B9
S-136_SA2_C1
S-448_SA2_C2
S-116_SA2_C3
S-113_SA2_C4
S-109_SA2_C5
S-149_SA2_C6
S-116_SA2_C7
S-190_SA2_C8
S-124_SA2_C9
S-107_SA2_D1
S-447_SA2_D2
S-181_SA2_D3
S-452_SA2_D4
S-148_SA2_D5
S-122_SA2_D6
S-192_SA2_D7
S-439_SA2_D8
S-483_SA2_D9
S-485_SA2_E1
S-464_SA2_E2
S-470_SA2_E3
S-456_SA2_E4
S-457_SA2_E5
S-467_SA2_E6
S-451_SA2_E7
S-475_SA2_E8
S-440_SA2_E9
S-499_SA2_F1
S-489_SA2_F2
S-442_SA2_F3
S-441_SA2_F4
S-434_SA2_F5
S-117_SA2_F6
S-449_SA2_F7
S-463_SA2_F8
S-128_SA2_F9
S-137_SA2_G1
S-461_SA2_G2
S-184_SA2_G3
S-138_SA2_G4
S-149_SA2_G5
S-110_SA2_G6
S-143_SA2_G7
S-151_SA2_G8
S-178_SA2_G9
S-188_SA2_H1
S-111_SA2_H2
S-145_SA2_H3
S-133_SA2_H4
S-146_SA2_H5
S-140_SA2_H6
S-112_SA2_H7
S-126_SA2_H8
S-112_SA2_H9
S-150_SA2_I1
S-178_SA2_I2
S-462_SA2_I3
S-453_SA2_I4
S-454_SA2_I5
S-443_SA2_I6
S-468_SA2_I7
S-469_SA2_I8
S-458_SA2_I9
S-445_SA3_A1
S-446_SA3_A2
S-120_SA3_A3
S-134_SA3_A4
S-150_SA3_A5
S-437_SA3_A6
S-125_SA3_A7
S-146_SA3_A8
S-438_SA3_A9
S-135_SA3_B1
S-115_SA3_B2
S-130_SA3_B3
S-126_SA3_B4
S-144_SA3_B5
S-187_SA3_B6
S-113_SA3_B7
S-112_SA3_B8
S-124_SA3_B9
S-187_SA3_C1
S-129_SA3_C2
S-113_SA3_C3
S-474_SA3_C4
S-499_SA3_C5
S-107_SA3_C6
S-123_SA3_C7
S-514_SA3_C8
S-438_SA3_C9
S-111_SA3_D1
S-113_SA3_D2
S-507_SA3_D3
S-138_SA3_D4
S-510_SA3_D5
S-183_SA3_D6
S-120_SA3_D7
S-504_SA3_D8
S-178_SA3_D9
S-144_SA3_E1
S-188_SA3_E2
S-136_SA3_E3
S-148_SA3_E4
S-187_SA3_E5
S-453_SA3_E6
S-470_SA3_E7
S-440_SA3_E8
S-452_SA3_E9
S-448_SA3_F1
S-186_SA3_F2
S-446_SA3_F3
S-442_SA3_F4
S-449_SA3_F5
S-454_SA3_F6
S-463_SA3_F7
S-444_SA3_F8
S-124_SA3_F9
S-450_SA3_G1
S-445_SA3_G2
S-458_SA3_G3
S-443_SA3_G4
S-451_SA3_G5
S-465_SA3_G6
S-434_SA3_G7
S-112_SA3_G8
S-512_SA3_G9
S-192_SA3_H1
S-432_SA3_H2
S-130_SA3_H3
S-505_SA3_H4
S-184_SA3_H5
S-509_SA3_H6
S-150_SA3_H7
S-508_SA3_H8
S-506_SA3_H9
S-135_SA3_I1
S-513_SA3_I2
S-114_SA3_I3
S-109_SA3_I4
S-128_SA3_I5
S-146_SA3_I6
S-115_SA3_I7
S-497_SA3_I8
S-185_SA3_I9
S-489_SA4_A1
S-191_SA4_A2
S-137_SA4_A3
S-116_SA4_A4
S-482_SA4_A5
S-126_SA4_A6
S-488_SA4_A7
S-437_SA4_A8
S-134_SA4_A9
S-452_SA4_B1
S-451_SA4_B2
S-450_SA4_B3
S-105_SA4_B4
S-497_SA4_B5
S-115_SA4_B6
S-113_SA4_B7
S-117_SA4_B9
S-120_SA4_C1
S-145_SA4_C2
S-500_SA4_C3
S-111_SA4_C4
S-129_SA4_C5
S-187_SA4_C6
S-192_SA4_C7
S-110_SA4_C8
S-141_SA4_C9
S-117_SA4_D1
S-484_SA4_D2
S-465_SA4_D3
S-443_SA4_D4
S-442_SA4_D5
S-457_SA4_D6
S-441_SA4_D7
S-464_SA4_D8
S-182_SA4_D9
S-507_SA4_E1
S-458_SA4_E2
S-448_SA4_E3
S-452_SA4_E4
S-445_SA4_E5
S-446_SA4_E6
S-449_SA4_E7
S-104_SA4_E8
S-113_SA4_E9
S-115_SA4_F1
S-120_SA4_F2
S-138_SA4_F3
S-145_SA4_F4
S-151_SA4_F5
S-181_SA4_F6
S-183_SA4_F7
S-497_SA4_F8
S-505_SA4_F9
S-510_SA4_G1
S-107_SA4_G2
S-109_SA4_G3
S-112_SA4_G4
S-114_SA4_G5
S-116_SA4_G6
S-126_SA4_G7
S-129_SA4_G8
S-136_SA4_G9
S-146_SA4_H1
S-148_SA4_H2
S-150_SA4_H3
S-187_SA4_H4
S-191_SA4_H5
S-482_SA4_H6
S-489_SA4_H7
S-500_SA4_H8
S-501_SA4_H9
S-502_SA4_I1
S-111_SA4_I2
S-123_SA4_I3
S-124_SA4_I4
S-130_SA4_I5
S-134_SA4_I6
S-137_SA4_I7
S-178_SA4_I8
S-184_SA4_I9
S-192_SA5_A1
S-485_SA5_A2
S-488_SA5_A3
S-506_SA5_A4
S-509_SA5_A5
S-511_SA5_A6
S-512_SA5_A7
S-513_SA5_A8
S-514_SA5_A9

'''

In [29]:
# Standardize f1 columns: extract prefix up to but NOT including _IP2_ (if present), otherwise up to the first _S or _R, then append the _sN suffix.
import re
def extract_prefix_and_suffix(col):
    # Find suffix _sN (case-insensitive)
    m_suffix = re.search(r'(_s\d+)$', col, flags=re.I)
    suffix = m_suffix.group(1).lower() if m_suffix else ''
    base = col[:m_suffix.start()] if m_suffix else col

    # If _IP2_ is present, prefix is everything up to but NOT including _IP2_
    m_ip2 = re.search(r'_IP2_', base)
    if m_ip2:
        prefix = base[:m_ip2.start()]
        return prefix + suffix

    # Otherwise, try to extract up to the first _S or _R (for S16, R1, etc.)
    m_sr = re.search(r'(_[SR]\d+)', base)
    if m_sr:
        prefix = base[:m_sr.start()]
        return prefix + suffix

    # Fallback: just use the base and suffix
    return base + suffix

# Apply to all columns in f1
f1_renamed = f1.copy()
new_names = [extract_prefix_and_suffix(col) for col in f1.columns]
f1_renamed.columns = new_names

# Show a preview of the renaming
print("First 20 original to new column names:")
for orig, new in list(zip(f1.columns, f1_renamed.columns))[:20]:
    print(f"{orig} -> {new}")

# Print how many renamings were successful (i.e., changed)
num_renamed = sum([orig != new for orig, new in zip(f1.columns, f1_renamed.columns)])
print(f"\nNumber of columns renamed: {num_renamed} out of {len(f1.columns)}")

First 20 original to new column names:
C-452-2_5_H3_IP2_1-11-Mar-22-Serum-P1B4-Y_S16_R1 -> C-452-2_5_H3
C-109-3_4_E7_IP2_1-2-Dec-20-Serum-P1F8-Y_S68_R1 -> C-109-3_4_E7
C-109-3_4_E7_IP2_2-2-Dec-20-Serum-P2F8-Y_S164_R1 -> C-109-3_4_E7
C-111-3_3_I6_IP2_1-2-Dec-20-Serum-P1C10-Y_S34_R1 -> C-111-3_3_I6
C-111-3_3_I6_IP2_2-2-Dec-20-Serum-P2C10-Y_S130_R1 -> C-111-3_3_I6
C-112-1_3_B3_IP2_1-12-Dec-18-Serum-P1F9-Y_S69_R1 -> C-112-1_3_B3
C-112-1_3_B3_IP2_2-12-Dec-18-Serum-P2F9-Y_S165_R1 -> C-112-1_3_B3
C-112-2_7_G7_IP2_1-3-Dec-20-Serum-P1B2-Y_S14_R1 -> C-112-2_7_G7
C-112-2_7_G7_IP2_2-3-Dec-20-Serum-P2B2-Y_S110_R1 -> C-112-2_7_G7
C-114-2_7_B1_IP2_1-22-Mar-Serum-P1E1-Y_S49_R1 -> C-114-2_7_B1
C-114-2_7_B1_IP2_2-22-Mar-Serum-P2E1-Y_S145_R1 -> C-114-2_7_B1
C-114-3_4_F3_IP2_1-2-Dec-20-Serum-P1C6-Y_S30_R1 -> C-114-3_4_F3
C-114-3_4_F3_IP2_2-2-Dec-20-Serum-P2C6-Y_S126_R1 -> C-114-3_4_F3
C-120-2_7_A4_IP2_1-22-Mar-Serum-P1H2-Y_S86_R1 -> C-120-2_7_A4
C-120-2_7_A4_IP2_2-22-Mar-Serum-P2H2-Y_S182_R1 -> C-120-2_7_

In [30]:
# Standardize f2 columns: extract prefix up to but NOT including the first _S or _R (after the ID), then append the _sN suffix. Leave the first column ("eptide") unchanged.
# Remove all trailing _N_ patterns (technical replicate) before the _sN suffix, even if there are multiple, and also if they are after the last _ (underscore).
import re
def extract_f2_prefix_and_suffix(col):
    if col == "eptide":
        return col
    # Find suffix _sN (case-insensitive) at the end
    m_suffix = re.search(r'(_s\d+)$', col, flags=re.I)
    suffix = m_suffix.group(1).lower() if m_suffix else ''
    base = col[:m_suffix.start()] if m_suffix else col

    # Remove all trailing _N_ patterns (technical replicate) before the suffix, even if after the last underscore
    base = re.sub(r'(_\d+)+$', '', base)

    # Remove a single trailing _N if present (e.g., ..._1)
    base = re.sub(r'_\d+$', '', base)

    # Try to extract up to the first _S or _R (for S24, R1, etc.)
    m_sr = re.search(r'(_[SR]\d+)', base)
    if m_sr:
        prefix = base[:m_sr.start()]
        return prefix + suffix
    # Fallback: just use the base and suffix
    return base + suffix

# Apply to all columns in f2
f2_renamed = f2.copy()
new_names_f2 = [extract_f2_prefix_and_suffix(col) for col in f2.columns]
f2_renamed.columns = new_names_f2

# Show a preview of the renaming
print("First 20 original to new column names in f2:")
for orig, new in list(zip(f2.columns, f2_renamed.columns))[:20]:
    print(f"{orig} -> {new}")

# Print how many renamings were successful (i.e., changed)
num_renamed_f2 = sum([orig != new for orig, new in zip(f2.columns, f2_renamed.columns)])
print(f"\nNumber of columns renamed in f2: {num_renamed_f2} out of {len(f2.columns)}")

First 20 original to new column names in f2:
AG_2_2_S118_R1 -> AG_2_2
C-109-1_5_B3_1_S24_R1 -> C-109-1_5_B3_1
C-109-1_5_B3_2_S120_R1 -> C-109-1_5_B3_2
C-113-1_7_B3_1_S53_R1 -> C-113-1_7_B3_1
C-114-2_4_H1_1_S20_R1 -> C-114-2_4_H1_1
C-114-2_4_H1_2_S116_R1 -> C-114-2_4_H1_2
C-114-3_3_B6_1_S46_R1 -> C-114-3_3_B6_1
C-114-3_3_B6_2_S142_R1 -> C-114-3_3_B6_2
C-114-3_7_A9_1_S19_R1 -> C-114-3_7_A9_1
C-114-3_7_A9_2_S115_R1 -> C-114-3_7_A9_2
C-115-3_1_D6_1_S25_R1 -> C-115-3_1_D6_1
C-115-3_1_D6_2_S121_R1 -> C-115-3_1_D6_2
C-122-2_3_B8_1_S71_R1 -> C-122-2_3_B8_1
C-122-2_3_B8_2_S167_R1 -> C-122-2_3_B8_2
C-126-1_4_A3_1_S74_R1 -> C-126-1_4_A3_1
C-126-1_4_A3_2_S170_R1 -> C-126-1_4_A3_2
C-126-3_6_I9_1_S76_R1 -> C-126-3_6_I9_1
C-126-3_6_I9_2_S172_R1 -> C-126-3_6_I9_2
C-128-1_5_B6_1_S66_R1 -> C-128-1_5_B6_1
C-128-1_5_B6_2_S162_R1 -> C-128-1_5_B6_2

Number of columns renamed in f2: 164 out of 164


In [31]:
# Standardize f3 columns: extract prefix up to but NOT including the first _S or _R (after the ID), then append the _sN suffix.
# Remove technical replicate (_N_) only if it is immediately before the _sN suffix, even if there are multiple such patterns.
import re
def extract_f3_prefix_and_suffix(col):
    # Find suffix _sN (case-insensitive) at the end
    m_suffix = re.search(r'(_s\d+)$', col, flags=re.I)
    suffix = m_suffix.group(1).lower() if m_suffix else ''
    base = col[:m_suffix.start()] if m_suffix else col

    # Remove all _N_ patterns immediately before the suffix (greedy, handles multiple)
    base = re.sub(r'(_\d+)+$', '', base)

    # Try to extract up to the first _S or _R (for S24, R1, etc.)
    m_sr = re.search(r'(_[SR]\d+)', base)
    if m_sr:
        prefix = base[:m_sr.start()]
        return prefix + suffix
    # Fallback: just use the base and suffix
    return base + suffix

# Apply to all columns in f3
f3_renamed = f3.copy()
#produces one string per column, then assigns it to the columns of f3_renamed
new_names_f3 = [extract_f3_prefix_and_suffix(col) for col in f3.columns]
f3_renamed.columns = new_names_f3

# Show a preview of the renaming
print("First 20 original to new column names in f3:")
for orig, new in list(zip(f3.columns, f3_renamed.columns))[:20]:
    print(f"{orig} -> {new}")

# Print how many renamings were successful (i.e., changed)
num_renamed_f3 = sum([orig != new for orig, new in zip(f3.columns, f3_renamed.columns)])
print(f"\nNumber of columns renamed in f3: {num_renamed_f3} out of {len(f3.columns)}")

First 20 original to new column names in f3:
AG_2_1_S108_R1 -> AG_2_1
AG_2_3_S133_R1 -> AG_2_3
C-110-2_6_A2_1_S49_R1 -> C-110-2_6_A2_1
C-110-2_6_A2_2_S145_R1 -> C-110-2_6_A2_2
C-113-3_1_G6_1_S58_R1 -> C-113-3_1_G6_1
C-113-3_1_G6_2_S154_R1 -> C-113-3_1_G6_2
C-114-1_7_B2_1_S53_R1 -> C-114-1_7_B2_1
C-114-1_7_B2_2_S149_R1 -> C-114-1_7_B2_2
C-115-1_7_A8_1_S52_R1 -> C-115-1_7_A8_1
C-115-1_7_A8_2_S148_R1 -> C-115-1_7_A8_2
C-116-2_5_D5_1_S13_R1 -> C-116-2_5_D5_1
C-116-2_5_D5_2_S109_R1 -> C-116-2_5_D5_2
C-120-1_4_G7_1_S2_R1 -> C-120-1_4_G7_1
C-120-1_4_G7_2_S98_R1 -> C-120-1_4_G7_2
C-124-3_3_C2_1_S62_R1 -> C-124-3_3_C2_1
C-124-3_3_C2_2_S158_R1 -> C-124-3_3_C2_2
C-125-3_3_E5_1_S67_R1 -> C-125-3_3_E5_1
C-125-3_3_E5_2_S163_R1 -> C-125-3_3_E5_2
C-126-1_1_E7_1_S61_R1 -> C-126-1_1_E7_1
C-126-1_1_E7_2_S157_R1 -> C-126-1_1_E7_2

Number of columns renamed in f3: 170 out of 170


In [32]:
# Standardize f4 columns: extract prefix up to but NOT including the first _S or _R (after the ID), then append the _sN suffix.
# Handles cases where there is an extra _s4 in the middle.
import re
def extract_f4_prefix_and_suffix(col):
    # Find suffix _sN (case-insensitive, at the end)
    m_suffix = re.search(r'(_s\d+)$', col, flags=re.I)
    suffix = m_suffix.group(1).lower() if m_suffix else ''
    base = col[:m_suffix.start()] if m_suffix else col

    # Try to extract up to the first _S or _R (for S24, R1, etc.)
    m_sr = re.search(r'(_[SR]\d+)', base)
    if m_sr:
        prefix = base[:m_sr.start()]
        return prefix + suffix

    # Fallback: just use the base and suffix
    return base + suffix

# Apply to all columns in f4
f4_renamed = f4.copy()
new_names_f4 = [extract_f4_prefix_and_suffix(col) for col in f4.columns]
f4_renamed.columns = new_names_f4

# Show a preview of the renaming
print("First 20 original to new column names in f4:")
for orig, new in list(zip(f4.columns, f4_renamed.columns))[:20]:
    print(f"{orig} -> {new}")

# Print how many renamings were successful (i.e., changed)
num_renamed_f4 = sum([orig != new for orig, new in zip(f4.columns, f4_renamed.columns)])
print(f"\nNumber of columns renamed in f4: {num_renamed_f4} out of {len(f4.columns)}")

First 20 original to new column names in f4:
AG_s4_1_S24_R1 -> AG_s4_1
AG_s4_2_S25_R1 -> AG_s4_2
AG_s5_1_S108_R1 -> AG_s5_1
AG_s5_2_S146_R1 -> AG_s5_2
AG_s5_4_S174_R1 -> AG_s5_4
C-110-1_6_A4_S99_R1 -> C-110-1_6_A4
C-110-3_2_E6_S157_R1 -> C-110-3_2_E6
C-110-3_5_E9_S154_R1 -> C-110-3_5_E9
C-111-2_5_B1_S153_R1 -> C-111-2_5_B1
C-111-3_6_F3_S33_R1 -> C-111-3_6_F3
C-112-2_2_A4_S15_R1 -> C-112-2_2_A4
C-112-2_7_B5_S140_R1 -> C-112-2_7_B5
C-113-1_1_D4_S170_R1 -> C-113-1_1_D4
C-113-2_7_B4_S35_R1 -> C-113-2_7_B4
C-113-3_6_F4_S60_R1 -> C-113-3_6_F4
C-115-1_1_I8_S135_R1 -> C-115-1_1_I8
C-115-3_5_F4_S129_R1 -> C-115-3_5_F4
C-116-1_7_A5_S97_R1 -> C-116-1_7_A5
C-116-2_2_I1_S63_R1 -> C-116-2_2_I1
C-116-2_7_D4_S122_R1 -> C-116-2_7_D4

Number of columns renamed in f4: 185 out of 185


In [33]:
# Standardize f5 columns: keep only the identifier up to but NOT including '-0-' and then the _sN suffix.
import re
def extract_f5_identifier_and_suffix(col):
    # Find the '-0-' pattern (not including it)
    m_id = re.search(r'^(.*?)-0-', col)
    # Find suffix _sN (case-insensitive, at the end)
    m_suffix = re.search(r'(_s\d+)$', col, flags=re.I)
    suffix = m_suffix.group(1).lower() if m_suffix else ''
    if m_id:
        identifier = m_id.group(1)
        return identifier + suffix
    # Fallback: just use the original column
    return col

# Apply to all columns in f5
f5_renamed = f5.copy()
new_names_f5 = [extract_f5_identifier_and_suffix(col) for col in f5.columns]
f5_renamed.columns = new_names_f5

# Show a preview of the renaming
print("First 25 original to new column names in f5:")
for orig, new in list(zip(f5.columns, f5_renamed.columns))[:30]:
    print(f"{orig} -> {new}")

# Print how many renamings were successful (i.e., changed)
num_renamed_f5 = sum([orig != new for orig, new in zip(f5.columns, f5_renamed.columns)])
print(f"\nNumber of columns renamed in f5: {num_renamed_f5} out of {len(f5.columns)}")

First 25 original to new column names in f5:
AG_1-0-AGBeads-P10B5-N_S401_R1 -> AG_1
AG_2-0-AGBeads-P9C8-N_S320_R1 -> AG_2
AG_4-0-AGBeads-P10G10-N_S466_R1 -> AG_4
C-109-2_5_B5-0-Plasma-P10E11-Y_S443_R1 -> C-109-2_5_B5
C-110-1_2_A5-0-Plasma-P6C9-Y_S33_R1 -> C-110-1_2_A5
C-110-2_3_F8-0-Plasma-P8A7-Y_S199_R1 -> C-110-2_3_F8
C-110-3_5_H7-0-Plasma-P7B8-Y_S116_R1 -> C-110-3_5_H7
C-111-1_2_H8-0-Plasma-P6B10-Y_S22_R1 -> C-111-1_2_H8
C-111-1_3_H9-0-Plasma-P9D1-Y_S325_R1 -> C-111-1_3_H9
C-111-1_6_F1-0-Plasma-P10B7-Y_S403_R1 -> C-111-1_6_F1
C-111-2_3_I5-0-Plasma-P10D11-Y_S431_R1 -> C-111-2_3_I5
C-111-2_6_F2-0-Plasma-P7E5-Y_S149_R1 -> C-111-2_6_F2
C-112-1_7_B6-0-Plasma-P11F5-Y_S545_R1 -> C-112-1_7_B6
C-112-1_7_G1-0-Plasma-P10E1-Y_S433_R1 -> C-112-1_7_G1
C-112-2_5_A2-0-Plasma-P10H2-Y_S470_R1 -> C-112-2_5_A2
C-112-3_7_D5-0-Plasma-P7C11-Y_S131_R1 -> C-112-3_7_D5
C-112-3_7_G2-0-Plasma-P6G8-Y_S80_R1 -> C-112-3_7_G2
C-113-1_4_D1-0-Plasma-P9D7-Y_S331_R1 -> C-113-1_4_D1
C-113-2_1_D1-0-Plasma-P6F7-Y_S67_R1 

In [36]:
# Standardize f6 columns: keep only the identifier up to but NOT including '-0-' and then the _sN suffix.
import re
def extract_f5_identifier_and_suffix(col):
    # Find the '-0-' pattern (not including it)
    m_id = re.search(r'^(.*?)-0-', col)
    # Find suffix _sN (case-insensitive, at the end)
    m_suffix = re.search(r'(_s\d+)$', col, flags=re.I)
    suffix = m_suffix.group(1).lower() if m_suffix else ''
    if m_id:
        identifier = m_id.group(1)
        return identifier + suffix
    # Fallback: just use the original column
    return col

# Apply to all columns in f5
f6_renamed = f6.copy()
new_names_f6 = [extract_f5_identifier_and_suffix(col) for col in f6.columns]
f6_renamed.columns = new_names_f6

# Show a preview of the renaming
print("First 25 original to new column names in f5:")
for orig, new in list(zip(f6.columns, f6_renamed.columns))[:30]:
    print(f"{orig} -> {new}")

# Print how many renamings were successful (i.e., changed)
num_renamed_f6 = sum([orig != new for orig, new in zip(f5.columns, f6_renamed.columns)])
print(f"\nNumber of columns renamed in f6: {num_renamed_f6} out of {len(f6.columns)}")



First 25 original to new column names in f5:
AG_1-0-Serum-P1B12-N_S24_R1 -> AG_1
AG_1-0-Serum-P2B12-N_S120_R1 -> AG_1
AG_2-0-Serum-P1D3-N_S39_R1 -> AG_2
AG_2-0-Serum-P2D3-N_S135_R1 -> AG_2
AG_3-0-Serum-P1E9-N_S57_R1 -> AG_3
AG_3-0-Serum-P2E9-N_S153_R1 -> AG_3
AG_4-0-Serum-P1G4-N_S76_R1 -> AG_4
Canary_1-0-Serum-P1H11-N_S95_R1 -> Canary_1
Canary_1-0-Serum-P2H11-N_S191_R1 -> Canary_1
Canary_2-0-Serum-P1H12-N_S96_R1 -> Canary_2
Canary_2-0-Serum-P2H12-N_S192_R1 -> Canary_2
HBDB-001-0-Serum-P1A1-Y_S1_R1 -> HBDB-001
HBDB-001-0-Serum-P2A1-Y_S97_R1 -> HBDB-001
HBDB-002-0-Serum-P1B1-Y_S13_R1 -> HBDB-002
HBDB-002-0-Serum-P2B1-Y_S109_R1 -> HBDB-002
HBDB-003-0-Serum-P1C1-Y_S25_R1 -> HBDB-003
HBDB-003-0-Serum-P2C1-Y_S121_R1 -> HBDB-003
HBDB-005-0-Serum-P1D1-Y_S37_R1 -> HBDB-005
HBDB-005-0-Serum-P2D1-Y_S133_R1 -> HBDB-005
HBDB-006-0-Serum-P1E1-Y_S49_R1 -> HBDB-006
HBDB-006-0-Serum-P2E1-Y_S145_R1 -> HBDB-006
HBDB-007-0-Serum-P1F1-Y_S61_R1 -> HBDB-007
HBDB-007-0-Serum-P2F1-Y_S157_R1 -> HBDB-007
HBDB-00

### Standardize HBDB

In [37]:
# Concatenate original DataFrames (old names)


"""
all = pd.concat([f1, f2, f3, f4, f5], axis=1)
all.to_csv('All_source_plates_mapped_rpk_NOT_column_names_standardized.csv')
"""

# Concatenate DataFrames with remapped (standardized) column names
all_remapped = pd.concat([f1_renamed, f2_renamed, f3_renamed, f4_renamed, f5_renamed, f6_renamed], axis=1)

import re

HBDB_FULL_MAP = {
    "HBDB-A1_1": "HBDB-001",
    "HBDB-A1_s1": "HBDB-001",
    "HBDB-A1_s5" : "HBDB-001",
    "HBDB-A1_s4" : "HBDB-001",
    "HBDB-A2_1": "HBDB-010",
    "HBDB-A2_s4": "HBDB-010",
    "HBDB-A2_s5": "HBDB-010",
    "HBDB-A3_1": "HBDB-021",
    "HBDB-A3_2": "HBDB-021",
    "HBDB-A4_1": "HBDB-030",
    "HBDB-A4_2": "HBDB-030",
    "HBDB-A5_1": "HBDB-038",
    "HBDB-A5_2": "HBDB-038",
    "HBDB-A6_1": "HBDB-047",
    "HBDB-A6_2": "HBDB-047",
    "HBDB-A7_1": "HBDB-055",
    "HBDB-A8_1": "HBDB-064",

    "HBDB-A1": "HBDB-001",
    "HBDB-A2": "HBDB-010",
    "HBDB-A3": "HBDB-021",
    "HBDB-A4": "HBDB-030",
    "HBDB-A5": "HBDB-038",
    "HBDB-A6": "HBDB-047",
    "HBDB-A7": "HBDB-055",
    "HBDB-A8": "HBDB-064",
}


def standardize_hbdb_token(x: str) -> str:
    x = str(x).strip('_')

    # exact match
    if x in HBDB_FULL_MAP:
        return HBDB_FULL_MAP[x]

    # prefix match (handles things like HBDB-A1_1_s4)
    m = re.match(r'^(HBDB-A\d+_\d+)', x, flags=re.I)
    if m and m.group(1) in HBDB_FULL_MAP:
        return HBDB_FULL_MAP[m.group(1)]

    return x

def standardize_column_name(col: str) -> str:
    parts = str(col).split("|")
    parts = [standardize_hbdb_token(p) for p in parts]
    return "|".join(parts)

all_remapped.columns = [standardize_column_name(c) for c in all_remapped.columns]


In [38]:
all_remapped_hbdb_cols = [col for col in all_remapped.columns if str(col).startswith('HBDB')]

print(all_remapped_hbdb_cols)

['HBDB-001', 'HBDB-001', 'HBDB-010', 'HBDB-010', 'HBDB-001', 'HBDB-010', 'HBDB-021', 'HBDB-021', 'HBDB-030', 'HBDB-038', 'HBDB-038', 'HBDB-047', 'HBDB-047', 'HBDB-055', 'HBDB-064', 'HBDB-001', 'HBDB-001', 'HBDB-002', 'HBDB-002', 'HBDB-003', 'HBDB-003', 'HBDB-005', 'HBDB-005', 'HBDB-006', 'HBDB-006', 'HBDB-007', 'HBDB-007', 'HBDB-008', 'HBDB-008', 'HBDB-009', 'HBDB-009', 'HBDB-010', 'HBDB-010', 'HBDB-012', 'HBDB-012', 'HBDB-014', 'HBDB-014', 'HBDB-015', 'HBDB-015', 'HBDB-016', 'HBDB-019', 'HBDB-019', 'HBDB-020', 'HBDB-020', 'HBDB-021', 'HBDB-021', 'HBDB-022', 'HBDB-022', 'HBDB-023', 'HBDB-023', 'HBDB-025', 'HBDB-025', 'HBDB-026', 'HBDB-026', 'HBDB-027', 'HBDB-027', 'HBDB-028', 'HBDB-028', 'HBDB-030', 'HBDB-031', 'HBDB-031', 'HBDB-032', 'HBDB-032', 'HBDB-033', 'HBDB-033', 'HBDB-034', 'HBDB-034', 'HBDB-035', 'HBDB-035', 'HBDB-037', 'HBDB-037', 'HBDB-038', 'HBDB-040', 'HBDB-040', 'HBDB-041', 'HBDB-042', 'HBDB-042', 'HBDB-043', 'HBDB-043', 'HBDB-044', 'HBDB-044', 'HBDB-045', 'HBDB-045', 'HB

### dedupe

In [39]:
# Ensure unique identifiers for duplicate column names (technical replicates) in all remapped DataFrames
from collections import Counter, defaultdict

def make_unique_columns(cols, target_prefixes):
    # Sort longest first so more-specific prefixes match before shorter ones
    sorted_targets = sorted(target_prefixes, key=len, reverse=True)

    canonical = []
    for col in cols:
        col_str = str(col)
        # Strip pandas auto-dedup suffixes (.1, .2, ...) added during pd.concat
        col_clean = re.sub(r'\.\d+$', '', col_str)
        # Match against the canonical target prefixes
        matched = next((t for t in sorted_targets if col_clean.startswith(t)), None)
        canonical.append(matched if matched else col_clean)

    # Number only true duplicates (_1, _2, ...); unique columns keep their name
    total_counts = Counter(canonical)
    seen_counts = defaultdict(int)
    new_cols = []

    for canon in canonical:
        seen_counts[canon] += 1
        if total_counts[canon] > 1:
            new_cols.append(f"{canon}_{seen_counts[canon]}")
        else:
            new_cols.append(canon)

    return new_cols

In [40]:
# Parse canonical target prefixes from the raw list defined above
target_prefixes = [line.strip() for line in target_prefixes_raw.strip().splitlines() if line.strip()]

orig_cols = list(map(str, all_remapped.columns))

all_remapped_deduped = make_unique_columns(all_remapped.columns, target_prefixes)

all_remapped.columns = all_remapped_deduped

all_remapped.to_csv('All_source_plates_mapped_rpk_column_names_standardized_unique_identifiers_for_duplicates_with_hbdb_plate_samples_dropped_less_100k_readcounts_071426.csv')

In [41]:
all_remapped.columns

Index(['C-452-2_5_H3_1', 'C-109-3_4_E7_1', 'C-109-3_4_E7_2', 'C-111-3_3_I6_1',
       'C-111-3_3_I6_2', 'C-112-1_3_B3_1', 'C-112-1_3_B3_2', 'C-112-2_7_G7_1',
       'C-112-2_7_G7_2', 'C-114-2_7_B1_1',
       ...
       'HBDB-109_1', 'HBDB-109_2', 'HBDB-111_1', 'HBDB-111_2', 'HBDB-115_1',
       'HBDB-115_2', 'mAb_1_7', 'mAb_1_8', 'mAb_2_7', 'mAb_2_8'],
      dtype='object', length=1408)